In [ ]:
"""
I have split data, but I can't use both the train and validation sets together with FastAI's DataBlock.
"""

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch import nn
import torch.optim as optim
import torchvision.models as models

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'functions')))
from dataset import ChestXrayDataset
from train import train
from evaluation import plot_results ,eval_on_metrics
from gradcam import get_heatmap_for_resnet

In [ ]:
IMAGE_PATH = "../archive/"
import glob

# Tüm alt klasörlerdeki jpg ve png dosyalarını alalım
image_paths = glob.glob(IMAGE_PATH + "**/images/*.[jp][pn]g", recursive=True)

print(f"Toplam {len(image_paths)} resim bulundu.")

In [ ]:
TRAIN_PATH = '../data/AP_PA_Train.xlsx'
TEST_PATH = '../data/AP_PA_Test.xlsx'
VAL_PATH = '../data/AP_PA_Validation.xlsx'

In [ ]:
num_classes = 2
EPOCHS = 30

In [ ]:
from fastai.vision.all import *
import pandas as pd
import os

# Excel dosyasını oku
df = pd.read_excel(TRAIN_PATH)
df['View Position'] = df['View Position'].map({'AP': 0, 'PA': 1})
df['image_path'] = df['Image Index'].apply(lambda x: next((p for p in image_paths if os.path.basename(p) == x), None))

# Geçersiz yolları filtrele
df = df[df['image_path'].notnull()].reset_index(drop=True)

# Label fonksiyonu
def label_func(row): return row['View Position']
def image_func(row): return row['image_path']

# DataBlock tanımı
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_x=image_func,
    get_y=label_func,
    splitter=RandomSplitter(seed=42),
    item_tfms=Resize(224),
    batch_tfms=aug_transforms()
)

# DataLoaders oluştur
dls = dblock.dataloaders(df, bs=64)

# Kaç sınıf var otomatik çıkar
print("Sınıf sayısı:", dls.c)
dls.show_batch(max_n=9)


In [ ]:
learn = vision_learner(dls, resnet50, metrics=accuracy)
learn.fine_tune(3)

interp = ClassificationInterpretation.from_learner(learn)
interp.print_classification_report()